# 🏥 Patient No-Show Prediction Model
## Train → Deploy in Azure ML → Operationalize

**Data Source:** [Kaggle Medical Appointment No Shows](https://www.kaggle.com/datasets/joniarroba/noshowappointments)  
- 110,527 real medical appointments from Brazil
- 14 features including health conditions, demographics, and scheduling info

**What This Notebook Does:**
1. Load the Kaggle no-show dataset
2. Engineer predictive features
3. Train a Logistic Regression model (CPU-friendly)
4. Evaluate model performance (AUC, Brier Score)
5. Save model for Azure ML deployment

**Business Context:**  
~20% of patients no-show. Predicting high-risk appointments enables proactive interventions (reminders, overbooking).

---
*Demo: From data to production ML with Azure ML*

## 📦 Step 1: Install & Import Dependencies

These packages work on **CPU compute** (no GPU required) - perfect for Fabric F64 environments.

In [ ]:
# =============================================================================
# INSTALL REQUIRED PACKAGES (Run this first in Azure ML compute!)
# =============================================================================
# This cell installs packages that may not be in the default Azure ML environment

%pip install scikit-learn pandas numpy matplotlib seaborn joblib azure-identity azure-ai-ml azure-storage-file-datalake --quiet

print("✅ Packages installed! You may need to restart the kernel after first install.")
print("   Kernel → Restart Kernel (if imports fail below)")

: 

In [ ]:
# Core ML libraries (all CPU-compatible)
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == "notebooks" else NOTEBOOK_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml_service.fabric_onelake import (
    build_onelake_abfss_uri,
    build_onelake_https_uri,
    format_onelake_env_summary,
    list_onelake_directory,
    read_csv_from_onelake,
)

print("✅ All libraries imported successfully")
print(f"📅 Notebook run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🎬 Demo Console Flow

Use the next code cell as your live presenter console. It prints a short runbook, shows whether Fabric connection variables are set, and tells you which notebook cells to run next.

This keeps the demo focused on one story: Azure ML notebook connects to Fabric OneLake, validates access, loads the training data, and retrains the model.

In [ ]:
# =============================================================================
# DEMO CONSOLE: presenter-friendly runbook for the live demo
# =============================================================================
from textwrap import dedent

def demo_line(char="=", width=78):
    print(char * width)

def demo_title(title):
    demo_line("=")
    print(title)
    demo_line("=")

def demo_section(title):
    print()
    demo_line("-")
    print(title)
    demo_line("-")

demo_title("LIVE DEMO: Azure ML Notebook + Fabric OneLake")

print("Goal")
print("Show that Azure ML can securely reach Fabric OneLake, load training data, and retrain the no-show model.")

demo_section("Current Notebook Status")
fabric_status = {
    "FABRIC_WORKSPACE_NAME": os.getenv("FABRIC_WORKSPACE_NAME"),
    "FABRIC_WORKSPACE_ID": os.getenv("FABRIC_WORKSPACE_ID"),
    "FABRIC_LAKEHOUSE_NAME": os.getenv("FABRIC_LAKEHOUSE_NAME"),
    "FABRIC_LAKEHOUSE_ID": os.getenv("FABRIC_LAKEHOUSE_ID"),
    "FABRIC_DATA_PATH": os.getenv("FABRIC_DATA_PATH"),
    "FABRIC_ONELAKE_ENDPOINT": os.getenv("FABRIC_ONELAKE_ENDPOINT"),
    "FABRIC_USE_GUIDS": os.getenv("FABRIC_USE_GUIDS"),
}
for key, value in fabric_status.items():
    print(f"{key:26} : {value}")

has_fabric_connection = bool(
    (fabric_status["FABRIC_WORKSPACE_ID"] or fabric_status["FABRIC_WORKSPACE_NAME"])
    and (fabric_status["FABRIC_LAKEHOUSE_ID"] or fabric_status["FABRIC_LAKEHOUSE_NAME"])
)

print()
print(f"Fabric connection configured : {has_fabric_connection}")

demo_section("Live Demo Script")
script = [
    "1. Run Cell 1 if packages are not installed on this compute.",
    "2. Run Cell 2 to import the notebook helpers.",
    "3. Run the Fabric connectivity cell to prove Azure ML can browse the lakehouse.",
    "4. Run the training-data load cell to read the CSV directly from OneLake.",
    "5. Continue with the training and evaluation cells.",
]
for step in script:
    print(step)

demo_section("Talk Track")
talk_track = dedent(
    """
    We already provisioned Azure ML and compute.
    The remaining step is secure data access from Fabric.
    This notebook authenticates with Azure identity and reads directly from OneLake.
    First we validate access to the lakehouse.
    Then we load the training data and retrain the no-show model.
    """
).strip()
print(talk_track)

if not has_fabric_connection:
    demo_section("Set Fabric Variables Before the Live Demo")
    print("Run a small setup cell like this before the connectivity check:")
    print()
    print(dedent(
        '''
        import os

        os.environ["FABRIC_WORKSPACE_ID"] = "your-fabric-workspace-guid"
        os.environ["FABRIC_LAKEHOUSE_ID"] = "your-lakehouse-guid"
        os.environ["FABRIC_DATA_PATH"] = "Files/noshow/KaggleV2-May-2016.csv"
        os.environ["FABRIC_USE_GUIDS"] = "true"
        os.environ["FABRIC_ONELAKE_ENDPOINT"] = "onelake.dfs.fabric.microsoft.com"
        '''
    ).strip())

demo_section("Expected Audience Proof Points")
proof_points = [
    "Azure ML notebook identity can reach Fabric data.",
    "The lakehouse path is visible from the notebook.",
    "Training data is loaded from OneLake, not from a local copy.",
    "The model retrains on Azure ML compute.",
]
for point in proof_points:
    print(f"- {point}")

## 📊 Step 2: Load Training Data from Fabric OneLake or Local Sample

**Preferred demo path:** read the training file directly from your Fabric Lakehouse in OneLake.  
**Fallback path:** use the repo sample CSV when you are working locally in VS Code.

**Notebook environment variables:**
- `FABRIC_WORKSPACE_NAME` or `FABRIC_WORKSPACE_ID`
- `FABRIC_LAKEHOUSE_NAME` or `FABRIC_LAKEHOUSE_ID`
- `FABRIC_DATA_PATH` (default: `Files/noshow/KaggleV2-May-2016.csv`)
- `FABRIC_ONELAKE_ENDPOINT` (default: `onelake.dfs.fabric.microsoft.com`)
- `FABRIC_USE_GUIDS=true` when you are using GUID-based URIs

**Why GUIDs matter:** the ABFSS driver does not accept spaces in workspace names, so GUIDs are the safest option for repeatable demos.

In [ ]:
# =============================================================================
# Load data - prefer Fabric OneLake when FABRIC_* settings are present
# =============================================================================
DATA_PATH = "../data"
KAGGLE_FILE = f"{DATA_PATH}/KaggleV2-May-2016.csv"

FABRIC_CONFIG = {
    "workspace": os.getenv("FABRIC_WORKSPACE_ID") or os.getenv("FABRIC_WORKSPACE_NAME"),
    "lakehouse": os.getenv("FABRIC_LAKEHOUSE_ID") or os.getenv("FABRIC_LAKEHOUSE_NAME"),
    "data_path": os.getenv("FABRIC_DATA_PATH", "Files/noshow/KaggleV2-May-2016.csv"),
    "endpoint": os.getenv("FABRIC_ONELAKE_ENDPOINT", "onelake.dfs.fabric.microsoft.com"),
    "use_guid": os.getenv("FABRIC_USE_GUIDS", "false").lower() == "true",
}

if FABRIC_CONFIG["workspace"] and FABRIC_CONFIG["lakehouse"]:
    print("📂 Loading from Microsoft Fabric OneLake...")
    env_summary = format_onelake_env_summary(
        [
            "FABRIC_WORKSPACE_NAME",
            "FABRIC_WORKSPACE_ID",
            "FABRIC_LAKEHOUSE_NAME",
            "FABRIC_LAKEHOUSE_ID",
            "FABRIC_DATA_PATH",
            "FABRIC_ONELAKE_ENDPOINT",
            "FABRIC_USE_GUIDS",
        ]
    )
    for key, value in env_summary.items():
        print(f"   {key}={value}")

    https_uri = build_onelake_https_uri(
        FABRIC_CONFIG["workspace"],
        FABRIC_CONFIG["lakehouse"],
        FABRIC_CONFIG["data_path"],
        endpoint=FABRIC_CONFIG["endpoint"],
        use_guid=FABRIC_CONFIG["use_guid"],
    )
    print(f"   🌐 OneLake HTTPS URI: {https_uri}")

    if FABRIC_CONFIG["use_guid"] or " " not in FABRIC_CONFIG["workspace"]:
        abfss_uri = build_onelake_abfss_uri(
            FABRIC_CONFIG["workspace"],
            FABRIC_CONFIG["lakehouse"],
            FABRIC_CONFIG["data_path"],
            endpoint=FABRIC_CONFIG["endpoint"],
            use_guid=FABRIC_CONFIG["use_guid"],
        )
        print(f"   🔗 OneLake ABFSS URI: {abfss_uri}")
    else:
        print("   ℹ️ Workspace name contains spaces; use FABRIC_*_ID values for ABFSS-style URIs.")

    df = read_csv_from_onelake(
        FABRIC_CONFIG["workspace"],
        FABRIC_CONFIG["lakehouse"],
        FABRIC_CONFIG["data_path"],
        endpoint=FABRIC_CONFIG["endpoint"],
        use_guid=FABRIC_CONFIG["use_guid"],
    )
    print("   ✅ Downloaded from OneLake")
elif os.path.exists(KAGGLE_FILE):
    print(f"📂 Loading from local file: {KAGGLE_FILE}")
    df = pd.read_csv(KAGGLE_FILE)
else:
    raise FileNotFoundError(
        "Local sample file not found and FABRIC_* notebook variables are not configured. "
        "Set the Fabric variables first, then rerun this cell."
    )

# Standardize column names
df.columns = df.columns.str.lower().str.replace('-', '_')

# Parse dates
df['scheduledday'] = pd.to_datetime(df['scheduledday'])
df['appointmentday'] = pd.to_datetime(df['appointmentday'])

# Feature engineering
df['lead_time_days'] = (df['appointmentday'] - df['scheduledday']).dt.days
df['lead_time_days'] = df['lead_time_days'].clip(lower=0)
df['day_of_week'] = df['appointmentday'].dt.dayofweek
df['hour_of_day'] = df['scheduledday'].dt.hour

# Create chronic conditions count
df['chronic_conditions'] = (
    df['hipertension'].astype(int) + 
    df['diabetes'].astype(int) + 
    df['alcoholism'].astype(int)
)

# Convert target: "No" = showed up, "Yes" = no-show
df['no_show'] = (df['no_show'] == 'Yes').astype(int)

# Clean age
df['age'] = df['age'].clip(lower=0, upper=115)

print(f"\n✅ Loaded {len(df):,} appointment records")
print(f"📅 Date range: {df['appointmentday'].min().date()} to {df['appointmentday'].max().date()}")
print(f"🏘️ Neighborhoods: {df['neighbourhood'].nunique()}")
print(f"👥 Unique patients: {df['patientid'].nunique():,}")

### 🌐 Step 2a: Validate the Fabric OneLake connection

Run this cell after setting the `FABRIC_*` variables. It lists the first entries under the lakehouse `Files` area so you can prove the Azure ML notebook can reach OneLake before training starts.

If this cell fails, the usual cause is permissioning: the notebook user or Azure ML managed identity has not been granted read access in Fabric yet.

In [ ]:
# =============================================================================
# Connectivity check: browse the Fabric lakehouse before training
# =============================================================================
if not FABRIC_CONFIG["workspace"] or not FABRIC_CONFIG["lakehouse"]:
    raise ValueError(
        "Set FABRIC_WORKSPACE_NAME/ID and FABRIC_LAKEHOUSE_NAME/ID before running this cell."
    )

print("🌐 Checking Fabric OneLake connectivity...")
files_listing = list_onelake_directory(
    FABRIC_CONFIG["workspace"],
    FABRIC_CONFIG["lakehouse"],
    directory="Files",
    endpoint=FABRIC_CONFIG["endpoint"],
    use_guid=FABRIC_CONFIG["use_guid"],
)

print("✅ Connected. First entries under lakehouse Files:")
for entry in files_listing[:15]:
    print(f"   - {entry}")

if len(files_listing) > 15:
    print(f"   ... and {len(files_listing) - 15} more")

print("\nTarget training file:")
print(
    build_onelake_https_uri(
        FABRIC_CONFIG["workspace"],
        FABRIC_CONFIG["lakehouse"],
        FABRIC_CONFIG["data_path"],
        endpoint=FABRIC_CONFIG["endpoint"],
        use_guid=FABRIC_CONFIG["use_guid"],
    )
)

In [ ]:
# Quick data exploration
print("📋 Data Schema:")
print(df.dtypes)
print("\n" + "="*50)
print("\n📊 Sample Records:")
df.head()

## 📁 Step 2b: Register Data as Azure ML Data Asset (Optional)

Register the dataset to Azure ML for:
- **Data lineage tracking** - Know which data trained which model
- **Reusability** - Reference the same data across experiments
- **Governance** - Centralized data catalog

> 💡 **Demo Tip:** Show this in Portal → Data → Data Assets to demonstrate enterprise data governance.

In [ ]:
# =============================================================================
# REGISTER DATA AS AZURE ML DATA ASSET
# =============================================================================
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import os

# Configuration - your Azure ML workspace
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP = "<your-resource-group>"
WORKSPACE_NAME = "<your-workspace-name>"

print("🔐 Connecting to Azure ML...")
credential = DefaultAzureCredential()
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"✅ Connected to workspace: {WORKSPACE_NAME}")

# Use Data Lake URL (works in Azure ML compute) or local file
if os.path.exists(KAGGLE_FILE):
    data_path = KAGGLE_FILE
    print(f"📁 Using local file: {data_path}")
else:
    # Use ADLS Gen2 URL for Azure ML compute
    data_path = "https://<your-storage-account>.dfs.core.windows.net/ml-data/noshow/KaggleV2-May-2016.csv"
    print(f"🌐 Using Data Lake URL: {data_path}")

# Register the data as an asset
data_asset = Data(
    name="noshow-appointments-kaggle",
    path=data_path,
    type=AssetTypes.URI_FILE,
    description="Kaggle Medical Appointment No Shows dataset (110K records from Brazil)",
    tags={
        "source": "azure-datalake",
        "dataset": "noshowappointments",
        "records": str(len(df)),
        "use_case": "no-show-prediction",
        "storage_account": "<your-storage-account>"
    }
)

print("📤 Registering data asset...")
registered_data = ml_client.data.create_or_update(data_asset)

print(f"\\n✅ Data registered to Azure ML!")
print(f"   Name:    {registered_data.name}")
print(f"   Version: {registered_data.version}")
print(f"   Path:    {registered_data.path}")
print(f"\\n📊 View in Portal: Data → Data Assets → {registered_data.name}")

## 📈 Step 3: Exploratory Data Analysis

Understanding the no-show distribution and key predictors before modeling.

**Business Insight:** Even a small reduction in no-shows has significant operational impact.

In [ ]:
# No-show rate analysis
no_show_rate = df['no_show'].mean()
total_appointments = len(df)
no_shows = df['no_show'].sum()

print("="*60)
print("📊 NO-SHOW STATISTICS")
print("="*60)
print(f"Total Appointments:     {total_appointments:,}")
print(f"No-Shows:               {no_shows:,}")
print(f"No-Show Rate:           {no_show_rate:.1%}")
print(f"Attended:               {total_appointments - no_shows:,}")
print("="*60)

# Business impact calculation (example costs)
avg_appointment_value = 150  # EUR
annual_appointments = total_appointments * 2  # Annualized estimate
potential_loss = no_shows * avg_appointment_value * 2

print(f"\n💰 ESTIMATED ANNUAL IMPACT")
print(f"Lost revenue from no-shows: €{potential_loss:,.0f}")
print(f"10% reduction would save:   €{potential_loss * 0.10:,.0f}")
print(f"20% reduction would save:   €{potential_loss * 0.20:,.0f}")

In [ ]:
# Visualization: No-show rates by key factors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('No-Show Rate Analysis by Key Factors', fontsize=14, fontweight='bold')

# 1. By Age Group
ax1 = axes[0, 0]
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 35, 50, 65, 120], 
                         labels=['0-18', '19-35', '36-50', '51-65', '65+'])
age_rates = df.groupby('age_group', observed=True)['no_show'].mean()
bars = ax1.bar(age_rates.index.astype(str), age_rates.values, color='steelblue')
ax1.set_xlabel('Age Group')
ax1.set_ylabel('No-Show Rate')
ax1.set_title('By Age Group')
ax1.axhline(y=no_show_rate, color='red', linestyle='--', label=f'Overall: {no_show_rate:.1%}')
ax1.legend()

# 2. By SMS Received (key intervention)
ax2 = axes[0, 1]
sms_rates = df.groupby('sms_received')['no_show'].mean()
ax2.bar(['No SMS', 'SMS Sent'], sms_rates.values, color=['coral', 'seagreen'])
ax2.set_ylabel('No-Show Rate')
ax2.set_title('By SMS Reminder')
ax2.axhline(y=no_show_rate, color='red', linestyle='--')
for i, rate in enumerate(sms_rates.values):
    ax2.text(i, rate + 0.01, f'{rate:.1%}', ha='center')

# 3. By Health Conditions
ax3 = axes[1, 0]
cond_rates = df.groupby('chronic_conditions')['no_show'].mean()
ax3.bar(cond_rates.index, cond_rates.values, color='mediumpurple')
ax3.set_xlabel('Number of Chronic Conditions')
ax3.set_ylabel('No-Show Rate')
ax3.set_title('By Chronic Conditions (Hypertension + Diabetes + Alcoholism)')
ax3.axhline(y=no_show_rate, color='red', linestyle='--')

# 4. By Day of Week
ax4 = axes[1, 1]
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_rates = df.groupby('day_of_week')['no_show'].mean()
ax4.bar([day_names[i] for i in day_rates.index], day_rates.values, color='teal')
ax4.set_xlabel('Day of Week')
ax4.set_ylabel('No-Show Rate')
ax4.set_title('By Day of Week')
ax4.axhline(y=no_show_rate, color='red', linestyle='--')

plt.tight_layout()
plt.show()

print("\n💡 KEY INSIGHTS:")
print(f"   • Younger patients (19-35) have higher no-show rates")
print(f"   • SMS reminders: {sms_rates[0]:.1%} no-show WITHOUT vs {sms_rates[1]:.1%} WITH")
print(f"   • Patients with chronic conditions show up MORE (invested in health)")

## 🔧 Step 4: Feature Engineering

**Features from Kaggle dataset:**

| Feature | Type | Description |
|---------|------|-------------|
| `age` | Numeric | Patient age |
| `scholarship` | Binary | Bolsa Família welfare program |
| `hipertension` | Binary | Has hypertension |
| `diabetes` | Binary | Has diabetes |
| `alcoholism` | Binary | Alcoholism indicator |
| `handcap` | 0-4 | Disability level |
| `sms_received` | Binary | Received SMS reminder |
| `lead_time_days` | Numeric | Days between scheduling and appointment |
| `day_of_week` | 0-6 | Monday=0 to Sunday=6 |
| `chronic_conditions` | 0-3 | Sum of health conditions |

In [ ]:
# Define features for the model
# Using features available in the Kaggle dataset

FEATURES = [
    'age',                  # Demographic - younger patients no-show more
    'scholarship',          # Socioeconomic - Bolsa Família welfare indicator
    'hipertension',         # Health condition
    'diabetes',             # Health condition
    'alcoholism',           # Health condition
    'handcap',              # Disability level (0-4)
    'sms_received',         # Intervention - received reminder
    'lead_time_days',       # Time between scheduling and appointment
    'day_of_week',          # Temporal pattern
    'chronic_conditions',   # Aggregated health complexity
]

TARGET = 'no_show'

# Prepare feature matrix and target
X = df[FEATURES].fillna(0)
y = df[TARGET].astype(int)

print("📊 Feature Summary:")
print("="*60)
for feat in FEATURES:
    print(f"  {feat:20} | mean: {X[feat].mean():8.3f} | std: {X[feat].std():8.3f}")
print("="*60)
print(f"\n🎯 Target distribution:")
print(f"  No-show (1): {y.sum():,} ({y.mean():.1%})")
print(f"  Attended (0): {(1-y).sum():,} ({1-y.mean():.1%})")

## 🧠 Step 5: Train the Model

Using **Logistic Regression** because:
- ✅ Works well on CPU (no GPU needed)
- ✅ Interpretable coefficients (explainable to clinicians)
- ✅ Fast inference (important for real-time scoring)
- ✅ Outputs calibrated probabilities (good for risk scoring)

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintain class balance
)

print(f"📊 Train/Test Split:")
print(f"  Training samples:   {len(X_train):,}")
print(f"  Test samples:       {len(X_test):,}")
print(f"  Train no-show rate: {y_train.mean():.1%}")
print(f"  Test no-show rate:  {y_test.mean():.1%}")

In [ ]:
# Feature scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
print("🚀 Training Logistic Regression model...")
import time
start_time = time.time()

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    solver='lbfgs'
)

model.fit(X_train_scaled, y_train)

training_time = time.time() - start_time
print(f"✅ Training completed in {training_time:.2f} seconds")
print(f"   (This runs fast on CPU - no GPU needed!)")

## 📏 Step 6: Evaluate Model Performance

Key metrics for healthcare no-show prediction:
- **AUC-ROC**: Overall discriminative ability (target: ≥ 0.75)
- **Brier Score**: Probability calibration (lower is better)
- **Precision/Recall**: Balance between false alarms and missed no-shows

In [ ]:
# Generate predictions
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Calculate metrics
auc_score = roc_auc_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

print("="*60)
print("📊 MODEL PERFORMANCE METRICS")
print("="*60)
print(f"AUC-ROC Score:    {auc_score:.4f}  {'✅ Good!' if auc_score >= 0.75 else '⚠️ Needs improvement'}")
print(f"Brier Score:      {brier:.4f}   (lower is better, <0.25 is good)")
print("="*60)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Attended', 'No-Show']))

In [ ]:
# Visualization: ROC Curve and Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'Model (AUC = {auc_score:.3f})')
ax1.plot([0, 1], [0, 1], 'r--', label='Random (AUC = 0.5)')
ax1.fill_between(fpr, tpr, alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Confusion Matrix
ax2 = axes[1]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Attended', 'No-Show'],
            yticklabels=['Attended', 'No-Show'])
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
ax2.set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

# Business interpretation
tn, fp, fn, tp = cm.ravel()
print(f"\n💼 BUSINESS INTERPRETATION:")
print(f"  True Positives (caught no-shows):     {tp:,}")
print(f"  False Positives (unnecessary alerts): {fp:,}")
print(f"  False Negatives (missed no-shows):    {fn:,}")
print(f"  True Negatives (correct attends):     {tn:,}")

In [ ]:
# Feature importance (coefficient analysis)
feature_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': model.coef_[0],
    'Abs_Coefficient': np.abs(model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=True)

plt.figure(figsize=(10, 6))
colors = ['green' if c < 0 else 'red' for c in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
plt.xlabel('Coefficient (Positive = Increases No-Show Risk)')
plt.title('Feature Importance: What Drives No-Shows?')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\n🔍 KEY DRIVERS OF NO-SHOWS:")
for _, row in feature_importance.sort_values('Abs_Coefficient', ascending=False).head(3).iterrows():
    direction = "increases" if row['Coefficient'] > 0 else "decreases"
    print(f"  • {row['Feature']}: {direction} no-show risk (coef: {row['Coefficient']:.3f})")

## 💾 Step 7: Save Model Artifacts

Saving the trained model and scaler for deployment to Azure ML.

In [ ]:
# Create outputs directory
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model and scaler
model_path = f"{OUTPUT_DIR}/model.joblib"
scaler_path = f"{OUTPUT_DIR}/scaler.joblib"
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

# Save metrics
metrics = {
    "auc_roc": float(auc_score),
    "brier_score": float(brier),
    "training_samples": int(len(X_train)),
    "test_samples": int(len(X_test)),
    "features": FEATURES,
    "model_type": "LogisticRegression",
    "trained_at": datetime.now().isoformat(),
    "no_show_rate_train": float(y_train.mean()),
    "no_show_rate_test": float(y_test.mean())
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save feature config (needed for inference)
feature_config = {
    "features": FEATURES,
    "target": TARGET,
    "prediction_threshold": 0.7  # Default threshold for high-risk flagging
}
with open(f"{OUTPUT_DIR}/feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print("✅ Model artifacts saved:")
print(f"   📦 {model_path}")
print(f"   📦 {scaler_path}")
print(f"   📦 {OUTPUT_DIR}/metrics.json")
print(f"   📦 {OUTPUT_DIR}/feature_config.json")

## ☁️ Step 8: Register Model to Azure ML

This step registers the model to Azure ML Model Registry for:
- **Version control**: Track all model versions
- **Deployment**: Deploy to batch/online endpoints
- **Governance**: Audit trail of what's in production

> ⚠️ **Note:** Uncomment the code below when running with Azure ML access configured.

In [ ]:
# =============================================================================
# AZURE ML REGISTRATION
# =============================================================================

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model

# Configuration - your Azure ML workspace
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP = "<your-resource-group>"
WORKSPACE_NAME = "<your-workspace-name>"

print("🔐 Connecting to Azure ML...")

# Connect to Azure ML
credential = DefaultAzureCredential()
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

print(f"✅ Connected to workspace: {WORKSPACE_NAME}")

# Register the model
model_registration = Model(
    name="noshow-logreg",
    path=OUTPUT_DIR,
    type="custom_model",
    description=f"Logistic Regression no-show prediction model trained on Kaggle data. AUC: {auc_score:.4f}",
    tags={
        "framework": "scikit-learn",
        "task": "binary-classification",
        "use_case": "patient-no-show",
        "dataset": "kaggle-noshowappointments",
        "auc": str(round(auc_score, 4)),
        "brier": str(round(brier, 4)),
        "features": ",".join(FEATURES)
    }
)

print("📤 Registering model...")
registered_model = ml_client.models.create_or_update(model_registration)

print(f"\n✅ Model registered to Azure ML!")
print(f"   Name:    {registered_model.name}")
print(f"   Version: {registered_model.version}")
print(f"   ID:      {registered_model.id}")

## ✅ Summary & Next Steps

### What We Achieved:
| Metric | Value | Target |
|--------|-------|--------|
| AUC-ROC | See above | ≥ 0.75 |
| Brier Score | See above | < 0.25 |
| Training Time | < 1 sec | CPU-friendly ✅ |

### Next Steps in the Pipeline:
1. **Deploy Online Endpoint** → Real-time scoring API (below)
2. **Deploy Batch Endpoint** → Daily scoring of upcoming appointments
3. **Set up Monitoring** → Data drift detection and alerts
4. **Power BI Dashboard** → "Today's No-Show Risk List" for planners
5. **CI/CD Pipeline** → GitHub Actions for automated deployment

---

## 🚀 Step 9: Deploy Online Endpoint

Deploy a **real-time REST API** for no-show risk scoring.

**Use cases:**
- HiX integration (score when appointment is booked)
- Power Apps / Power Automate workflows  
- Third-party scheduling systems

In [ ]:
# =============================================================================
# CREATE ONLINE ENDPOINT
# =============================================================================
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, CodeConfiguration, Environment

ENDPOINT_NAME = "noshow-online-endpoint"

# Create the endpoint
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Real-time no-show risk scoring API",
    auth_mode="key",  # Use API key authentication
    tags={
        "use_case": "patient-no-show",
        "model": "logistic-regression"
    }
)

print(f"🚀 Creating online endpoint: {ENDPOINT_NAME}...")
print("   (This may take 2-3 minutes)")

try:
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"✅ Endpoint created: {ENDPOINT_NAME}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️ Endpoint {ENDPOINT_NAME} already exists, will update deployment")
    else:
        raise e

In [ ]:
# =============================================================================
# CREATE DEPLOYMENT (using model scoring script)
# =============================================================================
import os

# Get absolute paths for deployment files
DEPLOY_DIR = os.path.abspath("../deployment/online")
CODE_DIR = os.path.join(DEPLOY_DIR, "src")
ENV_FILE = os.path.join(DEPLOY_DIR, "environment.yml")

print(f"📁 Code directory: {CODE_DIR}")
print(f"📁 Environment file: {ENV_FILE}")

# Delete existing failed deployment first
try:
    print("🗑️ Deleting old deployment if exists...")
    ml_client.online_deployments.begin_delete(name="blue", endpoint_name=ENDPOINT_NAME).result()
    print("   Old deployment deleted")
except Exception as e:
    print(f"   No existing deployment to delete")

# Define the deployment using the registered model
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=f"azureml:{registered_model.name}:{registered_model.version}",
    code_configuration=CodeConfiguration(
        code=CODE_DIR,
        scoring_script="score_online.py"
    ),
    environment=Environment(
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
        conda_file=ENV_FILE
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
    environment_variables={
        "LOG_LEVEL": "DEBUG",
        "PREDICTION_THRESHOLD": "0.7"
    }
)

print(f"\n🚀 Creating deployment 'blue' on endpoint: {ENDPOINT_NAME}...")
print("   (This typically takes 8-10 minutes with new environment)")

try:
    ml_client.online_deployments.begin_create_or_update(deployment).result()
    
    # Set 100% traffic to this deployment
    endpoint.traffic = {"blue": 100}
    ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    
    print(f"✅ Deployment complete!")
    print(f"\n📡 Endpoint URL:")
    endpoint_details = ml_client.online_endpoints.get(ENDPOINT_NAME)
    print(f"   {endpoint_details.scoring_uri}")
except Exception as e:
    print(f"❌ Deployment failed: {e}")
    print("\n📋 Getting deployment logs for debugging...")
    try:
        logs = ml_client.online_deployments.get_logs(
            name="blue", 
            endpoint_name=ENDPOINT_NAME, 
            lines=100
        )
        print(logs)
    except:
        print("   Could not retrieve logs")

In [ ]:
# =============================================================================
# TEST THE ENDPOINT
# =============================================================================
import json
import requests

# Test payload - sample appointment (young patient, no SMS reminder)
test_data = {
    "age": 25,
    "scholarship": 1,
    "hipertension": 0,
    "diabetes": 0,
    "alcoholism": 0,
    "handcap": 0,
    "sms_received": 0,
    "lead_time_days": 21,
    "day_of_week": 4,
    "chronic_conditions": 0
}

print("🧪 Testing endpoint with sample data...")
print(f"   Input: {json.dumps(test_data, indent=2)}")

# Get endpoint details and API key
endpoint_details = ml_client.online_endpoints.get(ENDPOINT_NAME)
keys = ml_client.online_endpoints.get_keys(ENDPOINT_NAME)

# Make request using requests library
headers = {
    "Authorization": f"Bearer {keys.primary_key}",
    "Content-Type": "application/json"
}

response = requests.post(
    endpoint_details.scoring_uri,
    headers=headers,
    json=test_data
)

# Parse response (may be string or dict)
raw_result = response.json()
if isinstance(raw_result, str):
    result = json.loads(raw_result)
else:
    result = raw_result

print(f"\n✅ Response:")
print(f"   No-Show Risk:  {result.get('no_show_risk', 'N/A')}")
print(f"   Risk %:        {result.get('no_show_risk_pct', 'N/A')}%")
print(f"   Risk Category: {result.get('risk_category', 'N/A')}")
print(f"   Risk Flag:     {result.get('risk_flag', 'N/A')}")

print(f"\n🔑 For external API calls:")
print(f"   Endpoint: {endpoint_details.scoring_uri}")
print(f"   API Key:  {keys.primary_key[:20]}... (truncated)")
print(f"\n📋 Example curl command:")
print(f'   curl -X POST "{endpoint_details.scoring_uri}" \\')
print(f'     -H "Authorization: Bearer <API_KEY>" \\')
print(f'     -H "Content-Type: application/json" \\')
print(f'     -d \'{json.dumps(test_data)}\'')